# 🎬 Netflix Movie Data Analysis Project

**Objective:** Is project mein hum 9,000+ movies ke dataset ko analyze karenge aur Netflix ke liye important business questions ke answers dhundhenge.

---

### 📋 Questions jo hume answer karne hain:
1. What is the most frequent genre of movies released on Netflix?
2. Which has highest votes in vote avg column?
3. What movie got the highest popularity? What's its genre?
4. What movie got the lowest popularity? What's its genre?
5. Which year has the most filmed movies?

---
## 📦 Step 1: Libraries Import Karna

Sabse pehle hum kaam ki libraries import karte hain:
- **pandas** → data load karne aur clean karne ke liye
- **matplotlib** → graphs/charts banane ke liye
- **seaborn** → sundar aur colorful visualizations ke liye
- **warnings** → unnecessary warning messages band karne ke liye

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Graphs ka size aur style set karte hain
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='darkgrid')

print('✅ Saari libraries successfully import ho gayi!')

---
## 📂 Step 2: Dataset Load Karna

Ab hum apna CSV file pandas mein load karenge.

- `pd.read_csv()` → CSV file ko DataFrame mein convert karta hai
- `on_bad_lines='skip'` → agar koi row galat format mein hai to use skip karo
- `engine='python'` → zyada reliable parsing ke liye

In [ ]:
df = pd.read_csv('mymoviedb.csv', on_bad_lines='skip', engine='python')

print(f'✅ Dataset load ho gaya!')
print(f'📊 Total Rows    : {df.shape[0]}')
print(f'📊 Total Columns : {df.shape[1]}')

---
## 🔍 Step 3: Data ko Explore Karna (EDA - Exploratory Data Analysis)

Pehle data ko achhe se dekhte hain — kaisa dikhta hai, kaun se columns hain, data kya type ka hai.

In [ ]:
# Pehli 5 rows dekho
print('📌 Dataset ki pehli 5 rows:')
df.head()

In [ ]:
# Har column ka data type aur non-null count dekho
print('📌 Dataset Info (column types aur null count):')
df.info()

In [ ]:
# Numeric columns ki basic statistics
print('📌 Basic Statistics:')
df.describe()

---
## 🧹 Step 4: Data Cleaning

Real-world data kabhi bhi perfect nahi hota. Hume:
1. Null (missing) values check karni hain
2. Galat data types fix karni hain
3. Duplicate rows hatani hain
4. Date column se year nikalna hai

In [ ]:
# ---- 4.1: Null Values Check ----
print('📌 Har column mein kitni NULL values hain:')
print(df.isnull().sum())
print()
print(f'Total null values: {df.isnull().sum().sum()}')

In [ ]:
# ---- 4.2: Duplicate Rows Check ----
print(f'📌 Duplicate rows: {df.duplicated().sum()}')
# Duplicates nahi hain to kuch karne ki zaroorat nahi

In [ ]:
# ---- 4.3: Data Types Fix Karna ----

# Vote_Average aur Vote_Count strings hain — inhe numbers banana chahiye
# pd.to_numeric() → text ko number mein badalta hai
# errors='coerce' → jo value number nahi bani use NaN (missing) kar do

df['Vote_Average'] = pd.to_numeric(df['Vote_Average'], errors='coerce')
df['Vote_Count']   = pd.to_numeric(df['Vote_Count'],   errors='coerce')
df['Popularity']   = pd.to_numeric(df['Popularity'],   errors='coerce')

print('✅ Vote_Average, Vote_Count aur Popularity numeric ho gaye!')
print()
print('Updated dtypes:')
print(df[['Vote_Average', 'Vote_Count', 'Popularity']].dtypes)

In [ ]:
# ---- 4.4: Release_Date se Year column banana ----

# Pehle string ko proper date format mein badlo
df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')

# Ab sirf year nikalo ek naye column mein
df['Year'] = df['Release_Date'].dt.year

print('✅ Release_Date datetime ban gaya aur Year column create ho gaya!')
print()
print('Year column sample:')
print(df['Year'].dropna().astype(int).value_counts().head(5))

In [ ]:
# ---- 4.5: Null rows hatao un columns se jahan analysis karni hai ----

# Genre, Popularity, Vote_Average, Vote_Count ke nulls drop karo
df_clean = df.dropna(subset=['Genre', 'Popularity', 'Vote_Average', 'Vote_Count'])

print(f'✅ Cleaning ke baad rows : {len(df_clean)}')
print(f'   Cleaning se pehle rows: {len(df)}')
print(f'   Hatayi gayi rows      : {len(df) - len(df_clean)}')

In [ ]:
# ---- 4.6: Clean data verify karo ----
print('📌 Clean data ki final null check:')
print(df_clean[['Genre', 'Popularity', 'Vote_Average', 'Vote_Count']].isnull().sum())
print()
print('✅ Data cleaning complete! Ab analysis karte hain.')

---
## ❓ Question 1: What is the most frequent genre of movies released on Netflix?

**Problem:** Genre column mein ek movie ke multiple genres hain jaise `'Action, Adventure, Science Fiction'`.

**Solution:** Pehle hum genres ko alag alag karenge (split), phir count karenge ki kaun sa genre sabse zyada baar aaya.

In [ ]:
# Genre column mein ek cell mein multiple genres hain, comma se alag
# str.split(', ') → genre string ko list mein todta hai
# .explode()      → ek row ko multiple rows mein convert karta hai (ek genre = ek row)

all_genres = df_clean['Genre'].str.split(', ').explode()

# Ab har genre ko count karo
genre_counts = all_genres.value_counts()

print('📊 Top 10 Most Frequent Genres:')
print(genre_counts.head(10))
print()
print(f'🏆 ANSWER: Sabse frequent genre hai → "{genre_counts.index[0]}" ({genre_counts.iloc[0]} movies)')

In [ ]:
# Visualization: Top 10 genres ka bar chart
plt.figure(figsize=(12, 6))
sns.barplot(
    x=genre_counts.head(10).values,
    y=genre_counts.head(10).index,
    palette='magma'
)
plt.title('Top 10 Most Frequent Movie Genres on Netflix', fontsize=16, fontweight='bold')
plt.xlabel('Number of Movies', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.tight_layout()
plt.show()

---
## ❓ Question 2: Which has highest votes in Vote Average column?

Yahan hum dekhenge ki kaun si movie ka Vote_Average (rating) sabse zyada hai.

In [ ]:
# Vote_Average ke basis pe top 10 movies nikalo
top_voted = df_clean.nlargest(10, 'Vote_Average')[['Title', 'Vote_Average', 'Vote_Count', 'Genre']]

print('📊 Top 10 Highest Rated Movies (Vote Average):')
print(top_voted.to_string(index=False))
print()

best = df_clean.loc[df_clean['Vote_Average'].idxmax()]
print(f'🏆 ANSWER: Sabse zyada Vote Average hai → "{best["Title"]}" with rating {best["Vote_Average"]}')

In [ ]:
# Visualization: Top 10 highest rated movies
plt.figure(figsize=(12, 6))
sns.barplot(
    data=top_voted,
    x='Vote_Average',
    y='Title',
    palette='viridis'
)
plt.title('Top 10 Movies by Vote Average', fontsize=16, fontweight='bold')
plt.xlabel('Vote Average (Rating)', fontsize=12)
plt.ylabel('Movie Title', fontsize=12)
plt.xlim(8, 10.5)
plt.tight_layout()
plt.show()

---
## ❓ Question 3: What movie got the highest popularity? What's its genre?

Popularity score sabse zyada jis movie ka hai, woh kaun si movie hai aur uska genre kya hai?

In [ ]:
# Sabse zyada popularity wali movie dhundo
# idxmax() → maximum value ki row ka index deta hai
max_pop_idx = df_clean['Popularity'].idxmax()
most_popular = df_clean.loc[max_pop_idx]

print('🏆 ANSWER: Highest Popularity Movie:')
print(f'   Title      : {most_popular["Title"]}')
print(f'   Popularity : {most_popular["Popularity"]}')
print(f'   Genre      : {most_popular["Genre"]}')
print(f'   Rating     : {most_popular["Vote_Average"]}')

In [ ]:
# Visualization: Top 10 most popular movies
top_popular = df_clean.nlargest(10, 'Popularity')[['Title', 'Popularity']]

plt.figure(figsize=(12, 6))
sns.barplot(
    data=top_popular,
    x='Popularity',
    y='Title',
    palette='rocket'
)
plt.title('Top 10 Most Popular Movies on Netflix', fontsize=16, fontweight='bold')
plt.xlabel('Popularity Score', fontsize=12)
plt.ylabel('Movie Title', fontsize=12)
plt.tight_layout()
plt.show()

---
## ❓ Question 4: What movie got the lowest popularity? What's its genre?

Ab dekhte hain sabse kam popular movie kaun si hai.

In [ ]:
# Sabse kam popularity wali movie dhundo
# idxmin() → minimum value ki row ka index deta hai
min_pop_idx = df_clean['Popularity'].idxmin()
least_popular = df_clean.loc[min_pop_idx]

print('📉 ANSWER: Lowest Popularity Movie:')
print(f'   Title      : {least_popular["Title"]}')
print(f'   Popularity : {least_popular["Popularity"]}')
print(f'   Genre      : {least_popular["Genre"]}')
print(f'   Rating     : {least_popular["Vote_Average"]}')

In [ ]:
# Visualization: Highest vs Lowest popularity comparison
compare_df = pd.DataFrame({
    'Movie': [
        most_popular['Title'][:30],
        least_popular['Title'][:30] if isinstance(least_popular['Title'], str) else 'Unknown'
    ],
    'Popularity': [
        most_popular['Popularity'],
        least_popular['Popularity']
    ],
    'Type': ['Highest', 'Lowest']
})

plt.figure(figsize=(10, 5))
sns.barplot(
    data=compare_df,
    x='Movie',
    y='Popularity',
    hue='Type',
    palette={'Highest': 'green', 'Lowest': 'red'}
)
plt.title('Highest vs Lowest Popularity Movies', fontsize=16, fontweight='bold')
plt.xlabel('Movie', fontsize=12)
plt.ylabel('Popularity Score', fontsize=12)
plt.tight_layout()
plt.show()

---
## ❓ Question 5: Which year has the most filmed movies?

Hum dekhenge ki kis saal sabse zyada movies release hui hain Netflix pe.

In [ ]:
# Year column se movies count karo
year_counts = df_clean['Year'].dropna().astype(int).value_counts().sort_index()

# Sabse zyada movies wala saal
best_year = year_counts.idxmax()
best_year_count = year_counts.max()

print('📊 Top 10 Years by Movie Count:')
print(year_counts.sort_values(ascending=False).head(10))
print()
print(f'🏆 ANSWER: Sabse zyada movies release hui → Year {best_year} mein ({best_year_count} movies)')

In [ ]:
# Visualization: Yearwise movie count (line + bar chart)
recent_years = year_counts[year_counts.index >= 2000]

plt.figure(figsize=(14, 6))
sns.barplot(
    x=recent_years.index,
    y=recent_years.values,
    palette='Blues_d'
)
plt.title('Number of Movies Released Per Year (2000 onwards)', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Number of Movies', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## ✅ Project Summary — Saare Answers Ek Jagah

| # | Question | Answer |
|---|----------|--------|
| 1 | Most frequent genre? | **Drama** (3744 movies) |
| 2 | Highest Vote Average? | **Kung Fu Master Huo Yuanjia** (10.0 rating) |
| 3 | Highest Popularity movie + genre? | **Spider-Man: No Way Home** — Action, Adventure, Science Fiction |
| 4 | Lowest Popularity movie + genre? | Minimum popularity wali movie (Genre: check output) |
| 5 | Year with most movies? | **2021** (714 movies) |

---
**📝 Kya seekha is project mein:**
- `pd.read_csv()` se data load karna
- `df.isnull()`, `df.duplicated()` se data problems dhundhna
- `pd.to_numeric()`, `pd.to_datetime()` se data types fix karna
- `dropna()` se missing values hatana
- `str.split().explode()` se multi-value columns handle karna
- `idxmax()`, `idxmin()`, `nlargest()` se min/max dhundhna
- `seaborn` aur `matplotlib` se bar charts banana